# CSTH Benchmark Exploratory Data Analysis

This notebook provides a production-ready exploratory data analysis (EDA) workflow for the CSTH Simulated Benchmark Dataset. It covers dataset loading, integrity checks, descriptive statistics, and visualization strategies to accelerate downstream modeling and monitoring efforts.

## 1. Environment Setup

The notebook relies on the standard scientific Python stack plus PyTorch for loading the provided `.pt` tensors.

```bash
pip install torch numpy pandas seaborn matplotlib scipy scikit-learn
```

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy import stats
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_DIR = Path('../data/raw')
TRAIN_PATH = DATA_DIR / 'train.pt'
VAL_PATH = DATA_DIR / 'val.pt'
TEST_PATH = DATA_DIR / 'test.pt'

for path in (TRAIN_PATH, VAL_PATH, TEST_PATH):
    if not path.exists():
        raise FileNotFoundError(f'Missing dataset split: {path}')

print('Using data directory:', DATA_DIR.resolve())

## 2. Utility Functions

Helper utilities standardize tensor-to-DataFrame conversion and consolidate repeated diagnostics.

In [ ]:
def load_split(path: Path) -> tuple[torch.Tensor, torch.Tensor]:
    '''Load a dataset split saved as a dict with keys `X` and `y`.'''
    blob = torch.load(path, map_location='cpu')
    if isinstance(blob, dict):
        X = blob.get('X')
        y = blob.get('y')
    else:
        # legacy format
        X, y = blob

    if X is None or y is None:
        raise KeyError(f'Split {path.name} must contain tensors `X` and `y`.')

    if X.ndim != 3:
        raise ValueError(f'Expected 3D tensor (samples, time, features); got shape {tuple(X.shape)}')

    if X.shape[0] != y.shape[0]:
        raise ValueError(f'Mismatched sample counts: X={X.shape[0]}, y={y.shape[0]}')

    return X.float(), y.long()


def tensor_to_frame(X: torch.Tensor, y: torch.Tensor, split: str) -> pd.DataFrame:
    '''Flatten a 3D tensor into a tabular representation with metadata columns.'''
    samples, time_steps, num_features = X.shape
    feature_names = ['cold_water_flow', 'tank_level', 'temperature']
    if num_features != len(feature_names):
        feature_names = [f'feature_{i}' for i in range(num_features)]

    arrays = []
    for idx in range(samples):
        sample = X[idx].numpy()
        frame = pd.DataFrame(sample, columns=feature_names)
        frame['time_index'] = np.arange(time_steps)
        frame['sample_id'] = idx
        frame['split'] = split
        frame['target'] = int(y[idx])
        arrays.append(frame)

    return pd.concat(arrays, ignore_index=True)


def summarize_split(X: torch.Tensor, y: torch.Tensor) -> dict:
    return {
        'samples': int(X.shape[0]),
        'time_steps': int(X.shape[1]),
        'features': int(X.shape[2]),
        'positive_fraction': float((y == 1).float().mean()),
    }


def describe_features(df: pd.DataFrame) -> pd.DataFrame:
    metrics = df.groupby('feature').agg(
        mean_value=('value', 'mean'),
        std_value=('value', 'std'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        skewness=('value', lambda x: stats.skew(x, bias=False)),
        kurtosis=('value', lambda x: stats.kurtosis(x, bias=False)),
    )
    return metrics.reset_index()

## 3. Load Dataset Splits

In [ ]:
train_X, train_y = load_split(TRAIN_PATH)
val_X, val_y = load_split(VAL_PATH)
test_X, test_y = load_split(TEST_PATH)

split_summary = {
    'train': summarize_split(train_X, train_y),
    'val': summarize_split(val_X, val_y),
    'test': summarize_split(test_X, test_y),
}

print(json.dumps(split_summary, indent=2))

### Split Distribution Overview

Check class balance and dimensional consistency for each dataset split.

In [ ]:
pd.DataFrame(split_summary).T

## 4. Aggregate Descriptive Statistics

Convert the tensor data into a long-form DataFrame to support vectorized aggregation and visualization across splits.

In [ ]:
train_df = tensor_to_frame(train_X, train_y, split='train')
val_df = tensor_to_frame(val_X, val_y, split='val')
test_df = tensor_to_frame(test_X, test_y, split='test')

full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
feature_long = (
    full_df.melt(
        id_vars=['sample_id', 'time_index', 'target', 'split'],
        value_vars=[col for col in full_df.columns if col not in {'sample_id', 'time_index', 'target', 'split'}]
    )
    .rename(columns={'variable': 'feature', 'value': 'value'})
)
feature_long.head()

### Feature-Level Summary Statistics

In [ ]:
describe_features(feature_long)

### Class Distribution

In [ ]:
class_distribution = (
    feature_long.drop_duplicates(subset=['split', 'sample_id']).groupby(['split', 'target']).size().unstack(fill_value=0)
)
class_distribution['positive_rate'] = class_distribution[1] / class_distribution.sum(axis=1)
class_distribution

## 5. Temporal Diagnostics

Visualize how each variable evolves through time across randomly sampled trajectories and across the entire population.

In [ ]:
def plot_sample(sample_idx: int, split: str = 'train', features: list[str] | None = None):
    source = {
        'train': (train_X, train_y),
        'val': (val_X, val_y),
        'test': (test_X, test_y),
    }[split]

    X, y = source
    if sample_idx >= X.shape[0]:
        raise IndexError(f'Sample index {sample_idx} out of bounds for {split} split with {X.shape[0]} samples')

    features = features or ['cold_water_flow', 'tank_level', 'temperature'][: X.shape[2]]

    fig, axes = plt.subplots(len(features), 1, sharex=True, figsize=(12, 3 * len(features)))
    if len(features) == 1:
        axes = [axes]

    for ax, feature, channel in zip(axes, features, range(len(features))):
        ax.plot(X[sample_idx, :, channel].numpy(), label=feature)
        ax.set_ylabel(feature)
        ax.legend(loc='upper right')

    axes[-1].set_xlabel('Time step')
    fig.suptitle(f"{split.capitalize()} sample {sample_idx} (target={int(y[sample_idx])})")
    plt.tight_layout()


plot_sample(0, split='train')

### Population-Level Trends

In [ ]:
agg_by_time = (
    feature_long.groupby(['split', 'target', 'feature', 'time_index'])['value'].agg(['mean', 'std']).reset_index()
)


def plot_population_trends(split: str, feature: str):
    subset = agg_by_time[(agg_by_time['split'] == split) & (agg_by_time['feature'] == feature)]
    plt.figure(figsize=(12, 4))
    sns.lineplot(data=subset, x='time_index', y='mean', hue='target', palette='deep')
    plt.fill_between(
        subset[subset['target'] == 0]['time_index'],
        subset[subset['target'] == 0]['mean'] - subset[subset['target'] == 0]['std'],
        subset[subset['target'] == 0]['mean'] + subset[subset['target'] == 0]['std'],
        alpha=0.2,
        color=sns.color_palette('deep')[0],
        label='Normal ±1σ',
    )
    plt.fill_between(
        subset[subset['target'] == 1]['time_index'],
        subset[subset['target'] == 1]['mean'] - subset[subset['target'] == 1]['std'],
        subset[subset['target'] == 1]['mean'] + subset[subset['target'] == 1]['std'],
        alpha=0.2,
        color=sns.color_palette('deep')[1],
        label='Fault ±1σ',
    )
    plt.title(f'Population dynamics — {split} split, {feature}')
    plt.xlabel('Time step')
    plt.ylabel(feature)
    plt.legend()
    plt.tight_layout()


plot_population_trends('train', 'temperature')

## 6. Feature Engineering Sandbox

This section highlights canonical transformations to accelerate downstream experimentation. Use the output to benchmark scaling strategies or dimensionality reduction.

In [ ]:
scaler = StandardScaler()

# Collapse the temporal dimension via global pooling as a baseline representation
train_flat = train_X.reshape(train_X.shape[0], -1).numpy()
val_flat = val_X.reshape(val_X.shape[0], -1).numpy()
test_flat = test_X.reshape(test_X.shape[0], -1).numpy()

scaler.fit(train_flat)
train_scaled = scaler.transform(train_flat)
val_scaled = scaler.transform(val_flat)
test_scaled = scaler.transform(test_flat)

print(
    f'Scaled representations -> train: {train_scaled.shape}, val: {val_scaled.shape}, test: {test_scaled.shape}'
)

### Correlation Heatmap

Inspect correlations of temporally pooled features to identify redundant signals or monitoring opportunities.

In [ ]:
corr = pd.DataFrame(train_scaled).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True)
plt.title('Correlation heatmap of pooled features (train split)')
plt.tight_layout()

## 7. Export Diagnostics

Persist summary statistics for reporting or pipeline validation checks.

In [ ]:
OUTPUT_DIR = Path('../results/eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_stats = describe_features(feature_long)
summary_path = OUTPUT_DIR / 'feature_summary.csv'
summary_stats.to_csv(summary_path, index=False)

class_path = OUTPUT_DIR / 'class_distribution.csv'
class_distribution.to_csv(class_path)

metadata_path = OUTPUT_DIR / 'split_summary.json'
metadata_path.write_text(json.dumps(split_summary, indent=2))

print('Wrote:')
for path in (summary_path, class_path, metadata_path):
    print(' -', path.resolve())

## 8. Next Steps

- Integrate these diagnostics into automated data quality checks for continuous monitoring.
- Compare fault detection models (e.g., KNN, 1D CNN, transformers) using the standardized splits.
- Track drift by re-running the notebook on new batches and diffing the exported metrics.